# 01 — Concat

Loads and concatenates raw Garmin export files into flat CSVs for downstream cleaning.

## Sources
- `DI-Connect-Aggregator/UDSFile_*.json` — 9 files, daily wellness summaries (Dec 2023 – May 2026)
- `DI-Connect-Fitness/maxk678@gmail.com_0_summarizedActivities.json` — workout log

## Outputs
- `data/daily_raw.csv` — 887 rows, 51 columns
- `data/activities_raw.csv` — 259 rows, 119 columns

In [1]:
import json
import pandas as pd
from pathlib import Path

In [2]:
# Paths
project_root = Path().resolve().parent
aggregator_dir = project_root / "DI-Connect-Aggregator"
fitness_dir = project_root / "DI-Connect-Fitness"

print(aggregator_dir)
print(fitness_dir)

C:\Users\maxk6\Documents\Projects\garmin-watch-activities-analysis\DI-Connect-Aggregator
C:\Users\maxk6\Documents\Projects\garmin-watch-activities-analysis\DI-Connect-Fitness


In [3]:
# Load and concatenate UDS daily summary files
uds_files = sorted(aggregator_dir.glob("UDSFile_*.json"))
print(f"Found {len(uds_files)} UDS files:")
for f in uds_files:
    print(f" {f.name}")

Found 9 UDS files:
 UDSFile_2023-12-04_2024-03-13.json
 UDSFile_2024-03-13_2024-06-21.json
 UDSFile_2024-06-21_2024-09-29.json
 UDSFile_2024-09-29_2025-01-07.json
 UDSFile_2025-01-07_2025-04-17.json
 UDSFile_2025-04-17_2025-07-26.json
 UDSFile_2025-07-26_2025-11-03.json
 UDSFile_2025-11-03_2026-02-11.json
 UDSFile_2026-02-11_2026-05-22.json


In [4]:
# Concatenate all UDS files into one DataFrame
records = []
for f in uds_files:
    with open(f, "r", encoding="utf-8") as file:
        data = json.load(file)
        records.extend(data)

df_daily = pd.DataFrame(records)

print(f"Total records: {len(df_daily)}")
print(f"Date range: {df_daily['calendarDate'].min()} to {df_daily['calendarDate'].max()}")
print(f"Columns: {len(df_daily.columns)}")
df_daily.head(2)

Total records: 887
Date range: 2023-12-10 to 2026-05-21
Columns: 51


,userProfilePK,calendarDate,uuid,durationInMilliseconds,totalKilocalories,activeKilocalories,bmrKilocalories,wellnessKilocalories,remainingKilocalories,wellnessTotalKilocalories,...,bodyBattery,minAvgHeartRate,maxAvgHeartRate,version,hydration,respiration,restingCaloriesFromActivity,bodyBatteryFeedback,isVigorousDay,dailyTotalFromEpochData
0,118214492,2023-12-10,4e8553a7ae324c178c333cb5e1fbf0f3,86400000,2028.0,26.0,2002.0,2028.0,2028.0,2028.0,...,"{'userProfilePK': 118214492, 'calendarDate': '...",58.0,102.0,80460002,"{'userProfilePK': 118214492, 'calendarDate': '...","{'userProfilePK': 118214492, 'calendarDate': '...",NaN,NaN,NaN,NaN
1,118214492,2023-12-11,91e77183de7e43d3aca94f2d63ef743a,86400000,2435.0,433.0,2002.0,2435.0,2435.0,2435.0,...,"{'userProfilePK': 118214492, 'calendarDate': '...",49.0,124.0,83280002,"{'userProfilePK': 118214492, 'calendarDate': '...","{'userProfilePK': 118214492, 'calendarDate': '...",104.0,"{'userProfilePk': 118214492, 'calendarDate': [...",NaN,NaN


In [11]:
df_activities = pd.DataFrame(activities_data[0]["summarizedActivitiesExport"])

print(f"Total activities: {len(df_activities)}")
print(f"Columns: {len(df_activities.columns)}")
df_activities.head(2)

Total activities: 259
Columns: 119


,activityId,uuidMsb,uuidLsb,name,activityType,userProfileId,timeZoneId,beginTimestamp,eventTypeId,rule,...,floorsDescended,jumpCount,strokes,avgStrokes,avgSwolf,avgStrokeDistance,avgSwimCadence,poolLength,avgStrokeCadence,maxStrokeCadence
0,22941249248,-5819353018142080347,-4722233205957546280,Yoga,yoga,118214492,149,1779222688000,9,subscribers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22928860541,-7853096344831573636,-5482180969434325806,Yoga,yoga,118214492,149,1779136323000,9,subscribers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Save to CSV
output_dir = project_root / "data"
output_dir.mkdir(exist_ok=True)

df_daily.to_csv(output_dir / "daily_raw.csv", index=False)
df_activities.to_csv(output_dir / "activities_raw.csv", index=False)

print("Saved:")
print(f"  daily_raw.csv — {len(df_daily)} rows")
print(f"  activities_raw.csv — {len(df_activities)} rows")

Saved:
  daily_raw.csv — 887 rows
  activities_raw.csv — 259 rows
